In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider = "groq"
)
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.0', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D3FF580C10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D3FF6D0290>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
response = model.invoke("who is Imran Khan?")
response.content

"Imran Khan is a Pakistani politician, former cricketer, and philanthropist who served as the 22nd Prime Minister of Pakistan from 2018 to 2022. He is the founder and chairman of the Pakistan Tehreek-e-Insaf (PTI) party, which is one of the largest and most influential political parties in Pakistan.\n\n**Early Life and Cricket Career**\n\nImran Khan was born on October 5, 1952, in Lahore, Pakistan. He came from a wealthy and influential family and was educated at Aitchison College in Lahore and later at the University of Oxford. Khan is a renowned cricketer and played for the Pakistan national team from 1971 to 1992. He is widely regarded as one of the greatest fast bowlers and all-rounders in cricket history. Khan led the Pakistani team to victory in the 1992 Cricket World Cup, which was a historic achievement for Pakistan.\n\n**Politics and Philanthropy**\n\nAfter retiring from cricket, Khan entered politics and founded the PTI party in 1996. He has been a vocal advocate for social j

In [5]:
from langchain.tools import tool
@tool
def get_weather(location:str) ->str:
    """Get the weather at the location"""
    return f"The weather at {location} is sunny"

model_with_tool = model.bind_tools([get_weather])

In [6]:
response = model_with_tool.invoke("How is the weahter in Boston?")
print(response)
for i in response.tool_calls:
    print(f"Tool: {i['name']}")
    print(f"Args: {i['args']}")



content='' additional_kwargs={'tool_calls': [{'id': 'x6ef7wgez', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 222, 'total_tokens': 236, 'completion_time': 0.04726921, 'completion_tokens_details': None, 'prompt_time': 0.033329025, 'prompt_tokens_details': None, 'queue_time': 0.387365323, 'total_time': 0.080598235}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f94b2-eb26-78a2-98ac-88e0653957e3-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'x6ef7wgez', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 222, 'output_tokens': 14, 'total_tokens': 236}
Tool: get_weather
Args: {'location': 'Boston'}


In [9]:
messages = [{"role":"user", "content": "what is the weather in Boston?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tool.invoke(messages)
print(final_response.text)

The weather in Boston is sunny.
